# 노트북 목표
1. 04번에서 학습한 제안 모델(`KcBERT PCA + Metadata + MLP`)을 baseline 모델과 같은 기준으로 비교한다.
2. PDF 기준 baseline인 `TF-IDF + Random Forest`, `KcBERT PCA only + MLP`, `Metadata only + MLP`를 학습한다.
3. 메타데이터의 조합을 총 8가지로 나눠, 어떤 메타데이터이 성능에 가장 큰 기여하는지를 확인한다.
4. 성능 상 우위의 모델을 저장한다.


## 1. 라이브러리 로드

- baseline 비교, MLP ablation, 성능 평가, 모델 저장에 필요한 패키지를 불러온다.
- 각 패키지의 역할은 다음과 같다.

1. `pandas` / `numpy`: 데이터프레임 처리와 수치 계산
2. `joblib` / `json`: 모델 bundle과 설정 파일 저장 및 로드
3. `TfidfVectorizer` / `RandomForestClassifier`: 전통적 텍스트 baseline 구성
4. `PCA` / `StandardScaler`: KcBERT 임베딩 축소와 메타데이터 정규화
5. `SMOTE` / `ImbPipeline`: 클래스 불균형 보정이 포함된 MLP pipeline 구성
6. `sklearn.metrics`: accuracy, F1, PR-AUC 등 공통 평가 지표 계산


In [ ]:
import json
from itertools import combinations

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 2. 데이터 로드 및 split 재현

- 02번에서 생성한 `reviews_embeddings_extract.csv`를 불러온다.
- 04번과 같은 random seed와 stratify 조건으로 train / validation / test split을 재현한다.
- 텍스트 baseline에는 `cleaned_review_text`를 사용하고, MLP 계열 모델에는 KcBERT 임베딩과 메타데이터를 사용한다.
- `rating`은 별점 정제 대상이므로 이벤트 판별 입력 feature에서 제외한다.


In [ ]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
MIN_TUNED_THRESHOLD = 0.5

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = ['text_length', 'emoji_count', 'photo_count']
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

text_train = raw_df.loc[train_idx, TEXT_COL].fillna('').astype(str)
text_val = raw_df.loc[val_idx, TEXT_COL].fillna('').astype(str)
text_test = raw_df.loc[test_idx, TEXT_COL].fillna('').astype(str)

print('Raw embedding data:', raw_df.shape)
print('Hybrid train:', X_train.shape)
print('Hybrid validation:', X_val.shape)
print('Hybrid test:', X_test.shape)
print('KcBERT feature 수:', len(emb_cols))
print('metadata feature:', meta_cols)
print('excluded feature:', ['rating'])
print('train label 분포:', y_train.value_counts().sort_index().to_dict())
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())

## 3. 공통 평가 함수 정의

- 모든 모델을 같은 기준으로 비교하기 위해 threshold 탐색과 성능 계산 함수를 공통으로 정의한다.
- 모델은 먼저 “이 리뷰가 이벤트 리뷰일 확률”을 출력한다.
- 이 확률이 threshold 이상이면 이벤트 리뷰(1), threshold 미만이면 일반 리뷰(0)로 판단한다.
- threshold 후보는 `0.5 이상`으로 제한하고, validation 데이터에서 F1-score가 가장 높은 값을 선택한다.
- 최종 비교표에는 선택된 threshold 기준의 validation/test 결과만 남긴다.
- validation에서 선택한 threshold만 test에 적용해 test 데이터가 모델 선택에 섞이는 것을 막는다.

### 평가 지표 설명
- Precision: 모델이 이벤트 리뷰라고 예측한 것 중 실제 이벤트 리뷰의 비율이다. 오탐이 많으면 낮아진다.
- Recall: 실제 이벤트 리뷰 중 모델이 이벤트 리뷰로 잡아낸 비율이다. 놓친 이벤트 리뷰가 많으면 낮아진다.
- F1-score: Precision과 Recall을 함께 반영한 균형 점수이다. 이 프로젝트에서는 이벤트 리뷰 탐지가 중요하므로 모델 선택의 1순위 기준으로 사용한다.
- PR-AUC: threshold를 여러 값으로 바꿨을 때 Precision과 Recall 관계를 종합한 값이다. 이벤트 리뷰처럼 관심 클래스가 적거나 불균형할 때 보조 지표로 유용하다.
- Accuracy: 전체 리뷰 중 맞춘 비율이다. 일반 리뷰가 더 많으면 높게 보일 수 있어 참고 지표로만 본다.
- ROC-AUC: 일반 리뷰와 이벤트 리뷰를 전반적으로 구분하는 능력을 보는 지표이다.
- FP: 실제 일반 리뷰인데 이벤트 리뷰로 잘못 예측한 개수이다.
- FN: 실제 이벤트 리뷰인데 일반 리뷰로 놓친 개수이다.

In [ ]:
def tune_threshold_from_validation(y_true, prob, min_threshold=0.5):
    precisions, recalls, thresholds = precision_recall_curve(y_true, prob)

    if len(thresholds) == 0:
        return float(min_threshold), pd.DataFrame()

    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    candidates = candidates[candidates['threshold'] >= min_threshold]

    if candidates.empty:
        return float(min_threshold), candidates

    best_threshold = float(candidates.sort_values('f1', ascending=False).iloc[0]['threshold'])
    return best_threshold, candidates.sort_values('f1', ascending=False).reset_index(drop=True)


def metric_row(model_name, feature_set, split, y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': model_name,
        'feature_set': feature_set,
        'split': split,
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, pred)),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


def evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob):
    best_threshold, threshold_candidates = tune_threshold_from_validation(
        y_val,
        val_prob,
        min_threshold=MIN_TUNED_THRESHOLD,
    )

    rows = [
        metric_row(model_name, feature_set, 'validation_tuned_min_0_5', y_val, val_prob, best_threshold),
        metric_row(model_name, feature_set, 'test_tuned_min_0_5', y_test, test_prob, best_threshold),
    ]
    return rows, best_threshold, threshold_candidates



def make_model_result_display(rows):
    display_df = pd.DataFrame(rows).copy()
    display_df = display_df[display_df['split'].isin([
        'validation_tuned_min_0_5',
        'test_tuned_min_0_5',
    ])]
    display_df['평가 데이터'] = display_df['split'].replace({
        'validation_tuned_min_0_5': 'Validation',
        'test_tuned_min_0_5': 'Test',
    })
    display_df = display_df.rename(columns={
        'threshold': 'threshold',
        'f1': 'F1',
        'pr_auc': 'PR-AUC',
        'precision': 'Precision',
        'recall': 'Recall',
        'accuracy': 'Accuracy',
        'roc_auc': 'ROC-AUC',
        'fp': 'FP(일반→이벤트)',
        'fn': 'FN(이벤트→일반)',
        'tp': 'TP(이벤트→이벤트)',
        'tn': 'TN(일반→일반)',
    })
    return display_df[[
        '평가 데이터',
        'threshold',
        'F1',
        'PR-AUC',
        'Precision',
        'Recall',
        'Accuracy',
        'ROC-AUC',
        'FP(일반→이벤트)',
        'FN(이벤트→일반)',
        'TP(이벤트→이벤트)',
        'TN(일반→일반)',
    ]]


def display_model_result(rows, title):
    print(title)
    display(make_model_result_display(rows).round({
        'threshold': 4,
        'F1': 4,
        'PR-AUC': 4,
        'Precision': 4,
        'Recall': 4,
        'Accuracy': 4,
        'ROC-AUC': 4,
    }))


## 4. 제안 모델 결과 로드

- 04번에서 저장한 `proposed_mlp_final_model.joblib`과 `proposed_mlp_selected_config.json`을 불러온다.
- 이 모델은 `KcBERT PCA + Metadata + MLP` 구조의 프로젝트 제안 모델이다.
- 05번에서는 제안 모델을 다시 학습하지 않고, 저장된 모델로 validation/test 성능을 같은 평가 함수에 맞춰 계산한다.


In [ ]:
with open('outputs/proposed_mlp_selected_config.json', 'r', encoding='utf-8') as f:
    proposed_config = json.load(f)

proposed_bundle_for_metrics = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model_for_metrics = proposed_bundle_for_metrics['model']
proposed_feature_cols = proposed_bundle_for_metrics['feature_cols']
proposed_threshold = float(
    proposed_bundle_for_metrics.get(
        'best_threshold',
        proposed_config.get('best_threshold_from_validation', 0.5),
    )
)

proposed_val_prob = proposed_model_for_metrics.predict_proba(X_val[proposed_feature_cols])[:, 1]
proposed_test_prob = proposed_model_for_metrics.predict_proba(X_test[proposed_feature_cols])[:, 1]
proposed_metrics = pd.DataFrame([
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'validation_tuned_min_0_5', y_val, proposed_val_prob, proposed_threshold),
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'test_tuned_min_0_5', y_test, proposed_test_prob, proposed_threshold),
])

display_model_result(proposed_metrics, '제안 모델(Hybrid MLP) 결과')


## 5. TF-IDF + Random Forest 베이스라인

- baseline 1에 해당하는 전통적 텍스트 모델이다.
- 입력은 `cleaned_review_text`만 사용한다.
- 텍스트는 TF-IDF unigram/bigram feature로 변환하고, 분류기는 `RandomForestClassifier`를 사용한다.
- KcBERT 임베딩이나 메타데이터 없이도 텍스트 표면 패턴만으로 어느 정도 이벤트 리뷰를 탐지하는지 확인한다.


In [ ]:
tfidf_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

tfidf_rf.fit(text_train, y_train)

tfidf_val_prob = tfidf_rf.predict_proba(text_val)[:, 1]
tfidf_test_prob = tfidf_rf.predict_proba(text_test)[:, 1]

tfidf_rows, tfidf_best_threshold, tfidf_threshold_candidates = evaluate_probabilities(
    'baseline_tfidf_random_forest',
    'cleaned_text_tfidf',
    y_val,
    tfidf_val_prob,
    y_test,
    tfidf_test_prob,
)

joblib.dump({
    'model': tfidf_rf,
    'model_name': 'baseline_tfidf_random_forest',
    'text_col': TEXT_COL,
    'target_col': LABEL_COL,
    'best_threshold': tfidf_best_threshold,
}, 'outputs/baseline_tfidf_random_forest_model.joblib')

display_model_result(tfidf_rows, 'TF-IDF + Random Forest 결과')


## 6. KcBERT only + MLP

- baseline 2에 해당하는 모델이다.
- 입력 feature는 KcBERT 임베딩 768차원만 사용한다.
- 04번에서 선택된 MLP 설정을 그대로 사용해, 제안 모델에서 메타데이터를 뺐을 때 성능이 어떻게 달라지는지 확인한다.


In [ ]:
def make_mlp_model(cols, use_emb, metadata_cols=None, random_state=SEED):
    metadata_cols = list(metadata_cols or [])
    transformers = []
    if use_emb:
        transformers.append((
            'kcbert_pca',
            PCA(n_components=proposed_config.get('pca_n_components', 0.90), random_state=random_state),
            emb_cols,
        ))
    if metadata_cols:
        transformers.append(('metadata_scaler', StandardScaler(), metadata_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    return ImbPipeline([
        ('preprocess', preprocessor),
        ('smote', SMOTE(
            random_state=random_state,
            k_neighbors=proposed_config.get('smote_k_neighbors', 5),
        )),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=tuple(proposed_config['hidden_layer_sizes']),
            activation=proposed_config.get('activation', 'relu'),
            solver=proposed_config.get('solver', 'adam'),
            alpha=proposed_config['alpha'],
            batch_size=proposed_config.get('batch_size', 64),
            learning_rate_init=proposed_config.get('learning_rate_init', 1e-3),
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


def fit_mlp_variant(model_name, feature_set, cols, use_emb, metadata_cols=None):
    cols = list(cols)
    metadata_cols = list(metadata_cols or [])
    X_train_variant = X_train[cols]
    X_val_variant = X_val[cols]
    X_test_variant = X_test[cols]

    model = make_mlp_model(cols, use_emb=use_emb, metadata_cols=metadata_cols, random_state=SEED)
    model.fit(X_train_variant, y_train)

    val_prob = model.predict_proba(X_val_variant)[:, 1]
    test_prob = model.predict_proba(X_test_variant)[:, 1]

    rows, best_threshold, threshold_candidates = evaluate_probabilities(
        model_name,
        feature_set,
        y_val,
        val_prob,
        y_test,
        test_prob,
    )
    return model, rows, best_threshold, threshold_candidates


pca_only_model, pca_only_rows, pca_only_threshold, pca_only_candidates = fit_mlp_variant(
    'ablation_mlp_kcbert_pca_only',
    'kcbert_pca_only',
    emb_cols,
    use_emb=True,
    metadata_cols=[],
)

joblib.dump({
    'model': pca_only_model,
    'model_name': 'ablation_mlp_kcbert_pca_only',
    'feature_cols': emb_cols,
    'emb_cols': emb_cols,
    'meta_cols': [],
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'best_threshold': pca_only_threshold,
    'selected_config': proposed_config,
}, 'outputs/ablation_mlp_kcbert_pca_only_model.joblib')

display_model_result(pca_only_rows, 'KcBERT only + MLP 결과')


## 7. Metadata only + MLP

- baseline 3에 해당하는 모델이다.
- 입력 feature는 `text_length`, `emoji_count`, `photo_count`만 사용한다.
- KcBERT 텍스트 임베딩 없이 행동 메타데이터만으로 이벤트 리뷰를 얼마나 구분할 수 있는지 확인한다.


In [ ]:
metadata_only_model, metadata_only_rows, metadata_only_threshold, metadata_only_candidates = fit_mlp_variant(
    'ablation_mlp_metadata_only',
    'metadata_only',
    meta_cols,
    use_emb=False,
    metadata_cols=meta_cols,
)

joblib.dump({
    'model': metadata_only_model,
    'model_name': 'ablation_mlp_metadata_only',
    'feature_cols': meta_cols,
    'emb_cols': [],
    'meta_cols': meta_cols,
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'best_threshold': metadata_only_threshold,
    'selected_config': proposed_config,
}, 'outputs/ablation_mlp_metadata_only_model.joblib')

display_model_result(metadata_only_rows, 'Metadata only + MLP 결과')


## 8. 메타데이터 경우의 수 조합 절제 실험

- 사용 가능한 메타데이터는 `text_length`, `emoji_count`, `photo_count` 3개이다.
- 이 3개 feature를 하나씩만 썼을 때, 두 개씩 조합했을 때, 세 개를 모두 썼을 때를 전부 비교한다.
- 가능한 조합은 총 7개이다.
  1. `text_length`
  2. `emoji_count`
  3. `photo_count`
  4. `text_length + emoji_count`
  5. `text_length + photo_count`
  6. `emoji_count + photo_count`
  7. `text_length + emoji_count + photo_count`
- 모든 조합은 같은 train/validation/test split, 같은 MLP 설정, 같은 threshold 탐색 규칙으로 평가한다.
- 모델 선택은 validation F1-score를 우선으로 보고, PR-AUC를 보조 기준으로 사용한다.
- Accuracy는 클래스 불균형 때문에 높게 보일 수 있으므로 참고용으로만 확인한다.
- split: 데이터를 train, validation, test로 나눈 구분이다.
- ablation: feature를 일부만 사용하거나 조합을 바꿔가며 어떤 입력이 성능에 도움이 되는지 확인하는 실험이다. (절제 실험)

In [ ]:
metadata_ablation_specs = []
for size in range(1, len(meta_cols) + 1):
    for feature_tuple in combinations(meta_cols, size):
        feature_cols = list(feature_tuple)
        feature_slug = '_'.join(feature_cols)
        feature_label = '+'.join(feature_cols)
        metadata_ablation_specs.append({
            'model': f'metadata_ablation_mlp_{feature_slug}',
            'feature_set': f'metadata_ablation:{feature_label}',
            'feature_cols': feature_cols,
            'path': f'outputs/metadata_ablation_mlp_{feature_slug}_model.joblib',
            'input_type': 'tabular',
        })

metadata_ablation_rows = []
metadata_ablation_registry = []

for spec in metadata_ablation_specs:
    model, rows, threshold, threshold_candidates = fit_mlp_variant(
        spec['model'],
        spec['feature_set'],
        spec['feature_cols'],
        use_emb=False,
        metadata_cols=spec['feature_cols'],
    )
    metadata_ablation_rows.extend(rows)
    metadata_ablation_registry.append({
        'model': spec['model'],
        'feature_set': spec['feature_set'],
        'path': spec['path'],
        'input_type': spec['input_type'],
    })

    joblib.dump({
        'model': model,
        'model_name': spec['model'],
        'feature_cols': spec['feature_cols'],
        'emb_cols': [],
        'meta_cols': spec['feature_cols'],
        'target_col': LABEL_COL,
        'input_type': 'tabular_raw',
        'best_threshold': threshold,
        'selected_config': proposed_config,
        'metadata_ablation': True,
    }, spec['path'])

metadata_ablation_metrics = pd.DataFrame(metadata_ablation_rows)
SELECTION_SPLIT = 'validation_tuned_min_0_5'
metadata_ablation_validation = (
    metadata_ablation_metrics[metadata_ablation_metrics['split'] == SELECTION_SPLIT]
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
metadata_ablation_validation.insert(0, 'rank', metadata_ablation_validation.index + 1)


def readable_feature_set(feature_set: str) -> str:
    return feature_set.replace('metadata_ablation:', '').replace('+', ' + ')


metadata_ablation_display = metadata_ablation_validation.copy()
metadata_ablation_display['메타데이터 조합'] = metadata_ablation_display['feature_set'].apply(readable_feature_set)
metadata_ablation_display['feature 수'] = metadata_ablation_display['메타데이터 조합'].apply(lambda value: len(value.split(' + ')))
metadata_ablation_display = metadata_ablation_display.rename(columns={
    'rank': '순위',
    'threshold': 'threshold',
    'f1': 'F1',
    'pr_auc': 'PR-AUC',
    'precision': 'Precision',
    'recall': 'Recall',
    'accuracy': 'Accuracy',
    'roc_auc': 'ROC-AUC',
})[[
    '순위',
    '메타데이터 조합',
    'feature 수',
    'threshold',
    'F1',
    'PR-AUC',
    'Precision',
    'Recall',
    'Accuracy',
    'ROC-AUC',
]]


round_cols = {
    'threshold': 4,
    'F1': 4,
    'PR-AUC': 4,
    'Precision': 4,
    'Recall': 4,
    'Accuracy': 4,
    'ROC-AUC': 4,
}

print(f'메타데이터 조합 수: {len(metadata_ablation_specs)}')
print('모델 선택 기준: validation F1 우선, PR-AUC 보조')
print('표 기준: validation 데이터에서 0.5 이상 threshold 중 F1이 가장 높은 기준')
print('Validation F1 기준 메타데이터 조합 순위')
display(metadata_ablation_display.round(round_cols))

best_row = metadata_ablation_validation.iloc[0]
worst_row = metadata_ablation_validation.iloc[-1]
full_metadata_row = metadata_ablation_validation[
    metadata_ablation_validation['feature_set'] == 'metadata_ablation:text_length+emoji_count+photo_count'
]

print('메타데이터 조합 인사이트')
print(f"- 최상 조합: {readable_feature_set(best_row['feature_set'])} (F1={best_row['f1']:.4f}, PR-AUC={best_row['pr_auc']:.4f})")
print(f"- 최하 조합: {readable_feature_set(worst_row['feature_set'])} (F1={worst_row['f1']:.4f}, PR-AUC={worst_row['pr_auc']:.4f})")
print('- Accuracy는 참고 지표이며, 최종 해석은 validation F1과 PR-AUC를 우선한다.')


## 9. 메타데이터 영향도 분석

- 04번 제안 모델에서 메타데이터 컬럼을 하나씩 섞어 permutation importance를 계산한다.
- 특정 메타데이터를 섞었을 때 F1-score가 크게 떨어지면, 해당 feature가 모델 예측에 더 민감하게 작동한 것으로 해석한다.
- 이 단계는 모델 선택 기준이 아니라 사후 해석용이다.
- permutation importance (순열 중요도): 특정 feature 값을 무작위로 섞어 성능이 얼마나 떨어지는지 보는 중요도 분석 방법이다.
- 기본 반복 횟수는 10으로 설정


In [ ]:
def metadata_permutation_importance(model, X_val, y_val, metadata_cols, threshold, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X_val)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_val, base_pred)

    rows = []
    for col in metadata_cols:
        drops = []
        for _ in range(repeats):
            X_perm = X_val.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled

            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            perm_f1 = f1_score(y_val, perm_pred)
            drops.append(base_f1 - perm_f1)

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })

    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


proposed_bundle = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model = proposed_bundle['model']
proposed_feature_cols = proposed_bundle['feature_cols']
proposed_threshold = proposed_bundle.get('best_threshold', proposed_config['best_threshold_from_validation'])

metadata_importance = metadata_permutation_importance(
    proposed_model,
    X_val[proposed_feature_cols],
    y_val,
    meta_cols,
    proposed_threshold,
    repeats=50,
)

metadata_importance_display = metadata_importance.copy()
metadata_importance_display.insert(0, 'rank', metadata_importance_display.index + 1)
metadata_importance_display = metadata_importance_display.rename(columns={
    'metadata': '메타데이터',
    'base_f1': '기준 F1',
    'mean_f1_drop': '평균 F1 감소폭',
    'std_f1_drop': 'F1 감소폭 표준편차',
    'repeats': '반복 횟수',
})
metadata_importance_display = metadata_importance_display.round({
    '기준 F1': 4,
    '평균 F1 감소폭': 4,
    'F1 감소폭 표준편차': 4,
})

top_metadata = metadata_importance.iloc[0]
print(
    f"가장 큰 영향을 준 메타데이터: {top_metadata['metadata']} "
    f"(평균 F1 감소폭: {top_metadata['mean_f1_drop']:.4f})"
)
display(metadata_importance_display)

## 10. 전체 비교표 출력 및 모델 저장

- 04번 제안 모델, TF-IDF baseline, KcBERT-only, metadata-only, metadata 조합 ablation 결과를 하나의 성능표로 합친다.
- 성능표는 노트북 화면에서 확인하고, 불필요한 CSV 파일로는 저장하지 않는다.
- 화면에는 각 모델 그룹에서 validation F1 기준 최고 결과만 요약해서 보여준다.
- 최종 선택 기준은 validation F1-score 우선, PR-AUC 보조 기준이다.
- test 결과는 선택된 모델의 참고 성능으로만 함께 표시한다.
- 06번이 후보 모델을 자동으로 읽을 수 있도록 `baseline_model_registry.csv`만 저장한다.


In [ ]:
baseline_metrics = pd.concat([
    proposed_metrics,
    pd.DataFrame(tfidf_rows),
    pd.DataFrame(pca_only_rows),
    pd.DataFrame(metadata_only_rows),
    metadata_ablation_metrics,
], ignore_index=True)

validation_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_summary.insert(0, 'rank', validation_summary.index + 1)

test_summary = (
    baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_summary.insert(0, 'rank', test_summary.index + 1)

validation_accuracy_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['accuracy', 'f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_accuracy_summary.insert(0, 'rank_accuracy', validation_accuracy_summary.index + 1)


saved_model_files = pd.DataFrame([
    {'model': 'baseline_tfidf_random_forest', 'feature_set': 'cleaned_text_tfidf', 'path': 'outputs/baseline_tfidf_random_forest_model.joblib', 'input_type': 'text'},
    {'model': 'ablation_mlp_kcbert_pca_only', 'feature_set': 'kcbert_pca_only', 'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib', 'input_type': 'tabular'},
    {'model': 'ablation_mlp_metadata_only', 'feature_set': 'metadata_only', 'path': 'outputs/ablation_mlp_metadata_only_model.joblib', 'input_type': 'tabular'},
    *metadata_ablation_registry,
    {'model': 'proposed_hybrid_mlp_04', 'feature_set': 'kcbert_pca+metadata', 'path': 'outputs/proposed_mlp_final_model.joblib', 'input_type': 'tabular'},
])
saved_model_files.to_csv('outputs/baseline_model_registry.csv', index=False, encoding='utf-8-sig')

validation_selection = baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_5'].copy()
test_reference = baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_5'].copy()

def readable_feature_label(feature_set):
    if feature_set.startswith('metadata_ablation:'):
        return readable_feature_set(feature_set)
    return {
        'cleaned_text_tfidf': 'cleaned_review_text TF-IDF',
        'kcbert_pca_only': 'KcBERT PCA only',
        'metadata_only': 'text_length + emoji_count + photo_count',
        'kcbert_pca+metadata': 'KcBERT PCA + metadata',
    }.get(feature_set, feature_set)

model_groups = [
    ('TF-IDF + Random Forest', validation_selection['model'].eq('baseline_tfidf_random_forest')),
    ('KcBERT only + MLP', validation_selection['model'].eq('ablation_mlp_kcbert_pca_only')),
    ('Metadata only + MLP', validation_selection['model'].eq('ablation_mlp_metadata_only')),
    ('Metadata ablation best', validation_selection['model'].str.startswith('metadata_ablation_mlp_')),
    ('제안 모델(Hybrid)', validation_selection['model'].eq('proposed_hybrid_mlp_04')),
]

best_rows = []
for group_name, mask in model_groups:
    group_candidates = validation_selection[mask]
    if group_candidates.empty:
        continue

    best_val = group_candidates.sort_values(['f1', 'pr_auc'], ascending=False).iloc[0]
    matching_test = test_reference[test_reference['model'] == best_val['model']]
    best_test = matching_test.iloc[0] if not matching_test.empty else None

    best_rows.append({
        '그룹': group_name,
        '선택 모델': best_val['model'],
        '입력 feature': readable_feature_label(best_val['feature_set']),
        'threshold': best_val['threshold'],
        'Validation F1': best_val['f1'],
        'Validation PR-AUC': best_val['pr_auc'],
        'Validation Precision': best_val['precision'],
        'Validation Recall': best_val['recall'],
        'Validation Accuracy': best_val['accuracy'],
        'Validation ROC-AUC': best_val['roc_auc'],
        'Test F1': np.nan if best_test is None else best_test['f1'],
        'Test PR-AUC': np.nan if best_test is None else best_test['pr_auc'],
        'Test Precision': np.nan if best_test is None else best_test['precision'],
        'Test Recall': np.nan if best_test is None else best_test['recall'],
        'Test Accuracy': np.nan if best_test is None else best_test['accuracy'],
        'Test ROC-AUC': np.nan if best_test is None else best_test['roc_auc'],
    })

best_by_group_summary = (
    pd.DataFrame(best_rows)
    .sort_values(['Validation F1', 'Validation PR-AUC'], ascending=False)
    .reset_index(drop=True)
)
best_by_group_summary.insert(0, '순위', best_by_group_summary.index + 1)
project_best = validation_summary.iloc[0]

round_cols = {
    'threshold': 4,
    'Validation F1': 4,
    'Validation PR-AUC': 4,
    'Validation Precision': 4,
    'Validation Recall': 4,
    'Validation Accuracy': 4,
    'Validation ROC-AUC': 4,
    'Test F1': 4,
    'Test PR-AUC': 4,
    'Test Precision': 4,
    'Test Recall': 4,
    'Test Accuracy': 4,
    'Test ROC-AUC': 4,
}

print('그룹별 최고 모델 요약')
print('기준: validation_tuned_min_0_5, F1 우선 + PR-AUC 보조')
display(best_by_group_summary.round(round_cols))

print('전체 후보 중 validation F1 기준 1위')
display(pd.DataFrame([{
    '선택 모델': project_best['model'],
    '입력 feature': readable_feature_label(project_best['feature_set']),
    'Validation F1': project_best['f1'],
    'Validation PR-AUC': project_best['pr_auc'],
    'Validation Precision': project_best['precision'],
    'Validation Recall': project_best['recall'],
    'Validation Accuracy': project_best['accuracy'],
    'Validation ROC-AUC': project_best['roc_auc'],
}]).round({
    'Validation F1': 4,
    'Validation PR-AUC': 4,
    'Validation Precision': 4,
    'Validation Recall': 4,
    'Validation Accuracy': 4,
    'Validation ROC-AUC': 4,
}))

print(f"저장된 모델 registry: outputs/baseline_model_registry.csv ({len(saved_model_files)}개 모델)")
